# Lecture 12: Transformer Models & Libraries
### NLP Course 2027

---

## Learning Outcomes
- Navigate the Hugging Face ecosystem
- Understand transformer tokenizers (WordPiece, BPE)
- Use the `pipeline()` API for common NLP tasks
- Load and inspect pretrained models

**Primary Reference:** *NLP with Transformers* (Tunstall et al.), Ch.1 & Ch.2

## 1. The Hugging Face Ecosystem

```
┌─────────────────────────────────────────────────────┐
│              Hugging Face Ecosystem                   │
│                                                       │
│  transformers   ← model architectures + weights      │
│  datasets       ← 10,000+ NLP datasets               │
│  tokenizers     ← fast, Rust-based tokenizers        │
│  accelerate     ← distributed/multi-GPU training     │
│  evaluate       ← metrics (accuracy, BLEU, ROUGE)    │
│  peft           ← parameter-efficient fine-tuning    │
│  Hub (hf.co)   ← 500,000+ models & datasets         │
└─────────────────────────────────────────────────────┘
```

The Hub hosts models from Google, Meta, Microsoft, Stanford, and the community.
You can browse, download, and share models with a single line of code.

In [1]:
# Install (run once)
# !pip install transformers datasets tokenizers

from transformers import pipeline
import transformers
import torch

print(f'transformers version: {transformers.__version__}')

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers version: 4.45.2


## 2. The `pipeline()` API

The simplest way to use pretrained models — handles tokenization, inference, and decoding automatically.

```python
pipe = pipeline(task, model=model_name)
result = pipe(input_text)
```

### Available Tasks
| Task | String | Example model |
|------|--------|---------------|
| Text classification | `'text-classification'` | `distilbert-base-uncased-finetuned-sst-2-english` |
| NER | `'ner'` | `dbmdz/bert-large-cased-finetuned-conll03-english` |
| Fill-mask | `'fill-mask'` | `bert-base-uncased` |
| Text generation | `'text-generation'` | `gpt2` |
| Summarization | `'summarization'` | `facebook/bart-large-cnn` |
| Translation | `'translation_en_to_fr'` | `Helsinki-NLP/opus-mt-en-fr` |
| Question Answering | `'question-answering'` | `deepset/roberta-base-squad2` |
| Zero-shot classification | `'zero-shot-classification'` | `facebook/bart-large-mnli` |

In [2]:
# Sentiment analysis
sentiment_pipe = pipeline('text-classification',
                          model='distilbert-base-uncased-finetuned-sst-2-english')

reviews = [
    'This NLP course is absolutely fantastic! I love every lecture.',
    'The homework was too difficult and confusing.',
    'It was okay, nothing special but not terrible either.'
]

results = sentiment_pipe(reviews)
for text, result in zip(reviews, results):
    print(f'Text:   {text[:60]}')
    print(f'Label:  {result["label"]}  Score: {result["score"]:.4f}')
    print()

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text:   This NLP course is absolutely fantastic! I love every lectur
Label:  POSITIVE  Score: 0.9999

Text:   The homework was too difficult and confusing.
Label:  NEGATIVE  Score: 0.9996

Text:   It was okay, nothing special but not terrible either.
Label:  POSITIVE  Score: 0.9159



In [3]:
# Named Entity Recognition
ner_pipe = pipeline('ner', model='dbmdz/bert-large-cased-finetuned-conll03-english',
                    aggregation_strategy='simple')

text = 'Elon Musk founded SpaceX in 2002 and acquired Twitter in 2022. The company is headquartered in Austin, Texas.'
entities = ner_pipe(text)

print(f'Text: {text}\n')
print(f'{"Entity":25s} {"Type":10s} {"Score"}')
print('-' * 50)
for ent in entities:
    print(f'{ent["word"]:25s} {ent["entity_group"]:10s} {ent["score"]:.4f}')

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: Elon Musk founded SpaceX in 2002 and acquired Twitter in 2022. The company is headquartered in Austin, Texas.

Entity                    Type       Score
--------------------------------------------------
Elon Musk                 PER        0.9974
SpaceX                    ORG        0.9990
Twitter                   ORG        0.9988
Austin                    LOC        0.9974
Texas                     LOC        0.9991


In [4]:
# Fill-mask: BERT predicts masked tokens
fill_mask = pipeline('fill-mask', model='bert-base-uncased')

sentences = [
    'NLP stands for Natural Language [MASK].',
    'The [MASK] is the basic unit of meaning in language.',
    'Transformers have revolutionized [MASK] learning.',
]
for sent in sentences:
    results = fill_mask(sent, top_k=3)
    print(f'Input: {sent}')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["sequence"]}')
    print()

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

Input: NLP stands for Natural Language [MASK].
  [0.771] nlp stands for natural language processing.
  [0.021] nlp stands for natural language learning.
  [0.014] nlp stands for natural language recognition.

Input: The [MASK] is the basic unit of meaning in language.
  [0.095] the word is the basic unit of meaning in language.
  [0.094] the verb is the basic unit of meaning in language.
  [0.031] the term is the basic unit of meaning in language.

Input: Transformers have revolutionized [MASK] learning.
  [0.293] transformers have revolutionized human learning.
  [0.174] transformers have revolutionized machine learning.
  [0.031] transformers have revolutionized language learning.



In [5]:
# Zero-shot classification (no task-specific training needed)
classifier = pipeline('zero-shot-classification',
                       model='facebook/bart-large-mnli')

text = 'The company reported record profits despite the economic downturn.'
candidate_labels = ['finance', 'sports', 'politics', 'technology', 'entertainment']

result = classifier(text, candidate_labels)
print(f'Text: {text}\n')
print('Classification scores:')
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f'  {label:15s} {score:.4f}  {bar}')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: The company reported record profits despite the economic downturn.

Classification scores:
  finance         0.6784  ████████████████████
  technology      0.1037  ███
  entertainment   0.0950  ██
  sports          0.0656  █
  politics        0.0573  █


In [6]:
# Question Answering
qa_pipe = pipeline('question-answering', model='deepset/roberta-base-squad2')

context = """
The transformer architecture was introduced by Vaswani et al. in the paper
'Attention Is All You Need' published in 2017. The key innovation was the
self-attention mechanism, which allows the model to weigh the importance of
different words when processing each word. BERT, released by Google in 2018,
was one of the first large pretrained transformer models and achieved
state-of-the-art results on 11 NLP tasks.
"""

questions = [
    'When was the transformer architecture introduced?',
    'What is the key innovation of transformers?',
    'Who released BERT?',
]
for q in questions:
    answer = qa_pipe(question=q, context=context)
    print(f'Q: {q}')
    print(f'A: {answer["answer"]}  (score: {answer["score"]:.4f})')
    print()

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Q: When was the transformer architecture introduced?
A: 2017  (score: 0.9670)

Q: What is the key innovation of transformers?
A: the
self-attention mechanism  (score: 0.5953)

Q: Who released BERT?
A: Google  (score: 0.9618)



## 3. Transformer Tokenizers

Tokenizers convert raw text to model-compatible integer IDs.

### Tokenization Pipeline
```
Raw text
   ↓ Normalization (lowercase, Unicode)
   ↓ Pre-tokenization (split on spaces, punctuation)
   ↓ Tokenization (subword algorithm: WordPiece / BPE / Unigram)
   ↓ Post-processing (add [CLS], [SEP], padding)
  Token IDs → model
```

### Subword Algorithms
| Algorithm | Models | Handles OOV |
|-----------|--------|-------------|
| WordPiece | BERT | Yes (via ##subwords) |
| BPE (Byte Pair Encoding) | GPT-2, RoBERTa | Yes |
| SentencePiece | T5, mBERT | Yes |

### Special Tokens
| Token | Purpose | BERT | RoBERTa |
|-------|---------|------|---------|
| CLS | Classification embedding | [CLS] | <s> |
| SEP | Sequence separator | [SEP] | </s> |
| PAD | Padding | [PAD] | <pad> |
| MASK | Masked token | [MASK] | <mask> |
| UNK | Unknown token | [UNK] | <unk> |

In [7]:
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "Let's tokenize this sentence, including contractions and sub-words!"

# Basic encoding
tokens = tokenizer.tokenize(text)
print(f'Tokens ({len(tokens)}): {tokens}')

ids = tokenizer.encode(text, add_special_tokens=True)
print(f'IDs: {ids}')

decoded = tokenizer.decode(ids)
print(f'Decoded: {decoded}')

Tokens (16): ['let', "'", 's', 'token', '##ize', 'this', 'sentence', ',', 'including', 'contraction', '##s', 'and', 'sub', '-', 'words', '!']
IDs: [101, 2292, 1005, 1055, 19204, 4697, 2023, 6251, 1010, 2164, 21963, 2015, 1998, 4942, 1011, 2616, 999, 102]
Decoded: [CLS] let ' s tokenize this sentence, including contractions and sub - words! [SEP]


In [8]:
# Batch encoding with padding and truncation
texts = [
    'Short text.',
    'This is a somewhat longer sentence that has more words in it.',
    'The transformer architecture is the backbone of modern NLP systems.',
]

encoded = tokenizer(texts,
                    padding=True,        # pad to longest in batch
                    truncation=True,     # truncate to max_length
                    max_length=30,
                    return_tensors='pt') # return PyTorch tensors

print('input_ids shape:      ', encoded['input_ids'].shape)
print('attention_mask shape: ', encoded['attention_mask'].shape)
print()
for i, text in enumerate(texts):
    ids = encoded['input_ids'][i]
    mask = encoded['attention_mask'][i]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f'Text {i+1}: {text[:40]}')
    print(f'  Tokens: {tokens}')
    print(f'  Mask:   {mask.tolist()}')
    print()

input_ids shape:       torch.Size([3, 15])
attention_mask shape:  torch.Size([3, 15])

Text 1: Short text.
  Tokens: ['[CLS]', 'short', 'text', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
  Mask:   [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Text 2: This is a somewhat longer sentence that 
  Tokens: ['[CLS]', 'this', 'is', 'a', 'somewhat', 'longer', 'sentence', 'that', 'has', 'more', 'words', 'in', 'it', '.', '[SEP]']
  Mask:   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Text 3: The transformer architecture is the back
  Tokens: ['[CLS]', 'the', 'transform', '##er', 'architecture', 'is', 'the', 'backbone', 'of', 'modern', 'nl', '##p', 'systems', '.', '[SEP]']
  Mask:   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]



In [9]:
# Understanding subword tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

words = ['unbelievable', 'tokenization', 'transformer', 'supercalifragilistic',
         'NLP', 'GPT', 'BERT', 'embeddings', '2024', 'COVID-19']

print(f'{"Word":25s} → Subwords')
print('-' * 60)
for word in words:
    subtokens = tokenizer.tokenize(word)
    print(f'{word:25s} → {subtokens}')

Word                      → Subwords
------------------------------------------------------------
unbelievable              → ['unbelievable']
tokenization              → ['token', '##ization']
transformer               → ['transform', '##er']
supercalifragilistic      → ['super', '##cal', '##if', '##rag', '##ilis', '##tic']
NLP                       → ['nl', '##p']
GPT                       → ['gp', '##t']
BERT                      → ['bert']
embeddings                → ['em', '##bed', '##ding', '##s']
2024                      → ['202', '##4']
COVID-19                  → ['co', '##vid', '-', '19']


In [10]:
# Loading and using a pretrained model for feature extraction
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

text = 'The bank by the river is very beautiful.'
encoded = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**encoded)

print('Model outputs:')
print(f'  last_hidden_state: {outputs.last_hidden_state.shape}  (batch, seq_len, hidden_size=768)')
print(f'  pooler_output:     {outputs.pooler_output.shape}  (batch, 768) - [CLS] embedding')

# [CLS] token as sentence embedding
cls_embedding = outputs.last_hidden_state[:, 0, :]  # first token = [CLS]
print(f'\n[CLS] embedding shape: {cls_embedding.shape}')
print(f'First 10 values: {cls_embedding[0, :10].numpy().round(4)}')

Model outputs:
  last_hidden_state: torch.Size([1, 11, 768])  (batch, seq_len, hidden_size=768)
  pooler_output:     torch.Size([1, 768])  (batch, 768) - [CLS] embedding

[CLS] embedding shape: torch.Size([1, 768])
First 10 values: [ 0.064  -0.1971 -0.4115 -0.3233  0.002  -0.6097  0.2397  1.3151 -0.2893
 -0.8343]


## Practice Exercises

See **`Lecture-12-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Component | Tool | Key Concept |
|-----------|------|-------------|
| Pretrained models | `AutoModel` | Load any architecture |
| Tokenizers | `AutoTokenizer` | Subword tokenization |
| Quick inference | `pipeline()` | Zero-code task solving |
| Hub | huggingface.co | 500k+ models |

**Next Lecture**: Fine-Tuning Transformers — adapting BERT/RoBERTa to your specific task.

---
*Book reference: NLP with Transformers Ch.1, 2*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**